# N-gram Explorer

**NLP Traditional Preprocessing - N-gram Feature Comparison**

## Project Overview

Compare unigram, bigram, trigram, and combined n-gram features for text classification. Measure impact on feature space and accuracy.

## Learning Objectives

1. Understand n-grams and word context capture.
2. Compare feature spaces.
3. Measure classification impact.
4. Learn why (1,2) usually wins.
5. Analyze discriminative n-grams.

## Problem Statement

Which n-gram configuration produces the best text classification features?

## Why This Project Matters

- Unigrams miss negation.
- Bigrams capture context.
- Trigrams add diminishing returns.
- Understanding trade-offs builds efficient pipelines.

## Dataset Overview

IMDB reviews, 5K subset, binary sentiment.

## Dataset Source and License Notes

Hugging Face `imdb`. Free. Auto-download.

## Environment Setup

In [ ]:
import subprocess,sys,importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","scikit-learn","datasets"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Ready.")

## Imports

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns,warnings
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from datasets import load_dataset
warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
%matplotlib inline
print("Loaded.")

## Configuration / Constants

In [ ]:
SEED=42; N_SAMPLES=5000; TEST_SIZE=0.2; MAX_FEATURES=15000
CLASS_NAMES=["Negative","Positive"]; np.random.seed(SEED)

## Dataset Download or Loading

In [ ]:
ds=load_dataset("imdb",split="train")
df=ds.to_pandas().sample(n=N_SAMPLES,random_state=SEED).reset_index(drop=True)
print(f"Shape:{df.shape}")

## Data Validation Checks

In [ ]:
df=df.drop_duplicates(subset=["text"]).reset_index(drop=True)
assert df["label"].isin([0,1]).all()
df["word_count"]=df["text"].str.split().apply(len)
print("Validation passed.")

## Exploratory Data Analysis

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df["word_count"],bins=50,color="steelblue",edgecolor="white"); axes[0].set_title("Word Count")
for l,n in enumerate(CLASS_NAMES): axes[1].hist(df[df["label"]==l]["word_count"],bins=30,alpha=0.5,label=n)
axes[1].set_title("Words by Sentiment"); axes[1].legend()
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
cc=df["label"].value_counts().sort_index()
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(CLASS_NAMES,cc.values,color=["#C44E52","#55A868"]); ax.set_title("Sentiment")
plt.tight_layout(); plt.show()

## Train/Validation/Test Split Strategy

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df["text"],df["label"],test_size=TEST_SIZE,random_state=SEED,stratify=df["label"])
print(f"Train:{len(X_train)}|Test:{len(X_test)}")

## Preprocessing Strategy - N-gram Configurations

In [ ]:
NGRAM_CONFIGS={"Unigram (1,1)":(1,1),"Bigram (1,2)":(1,2),"Trigram (1,3)":(1,3),"Bigram only (2,2)":(2,2),"Trigram only (3,3)":(3,3)}

## Feature Engineering - Feature Space Analysis

In [ ]:
fs=[]
for n,ng in NGRAM_CONFIGS.items():
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=ng)
    X_tr=tfidf.fit_transform(X_train)
    fs.append({"Config":n,"Features":X_tr.shape[1],"Nonzero/Doc":round(X_tr.nnz/X_tr.shape[0],1)})
fdf=pd.DataFrame(fs); print(fdf.to_string(index=False))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
colors=["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
axes[0].bar(fdf["Config"],fdf["Features"],color=colors); axes[0].set_title("Features"); axes[0].tick_params(axis="x",rotation=30)
axes[1].bar(fdf["Config"],fdf["Nonzero/Doc"],color=colors); axes[1].set_title("Nonzero/Doc"); axes[1].tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()

## Baseline Model

In [ ]:
tfidf_b=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,1))
X_tr_b=tfidf_b.fit_transform(X_train); X_te_b=tfidf_b.transform(X_test)
bl=MultinomialNB(); bl.fit(X_tr_b,y_train); y_pb=bl.predict(X_te_b)
bl_acc=accuracy_score(y_test,y_pb); bl_f1=f1_score(y_test,y_pb)
print(f"BASELINE: Acc={bl_acc:.4f}, F1={bl_f1:.4f}")
print(classification_report(y_test,y_pb,target_names=CLASS_NAMES))

## Full N-gram x Classifier Comparison

In [ ]:
classifiers={"MultinomialNB":MultinomialNB(),"LogisticRegression":LogisticRegression(max_iter=1000,random_state=SEED),"LinearSVC":LinearSVC(max_iter=2000,random_state=SEED)}
results=[]
for nn,ng in NGRAM_CONFIGS.items():
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=ng)
    X_tr=tfidf.fit_transform(X_train); X_te=tfidf.transform(X_test)
    for cn,clf in classifiers.items():
        m=clone(clf); m.fit(X_tr,y_train); yp=m.predict(X_te)
        results.append({"N-gram":nn,"Classifier":cn,"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp)})
results_df=pd.DataFrame(results)
print(results_df.sort_values("F1",ascending=False).to_string(index=False))

In [ ]:
pivot=results_df.pivot(index="N-gram",columns="Classifier",values="F1")
fig,ax=plt.subplots(figsize=(9,6))
sns.heatmap(pivot,annot=True,fmt=".4f",cmap="YlGnBu",ax=ax)
ax.set_title("F1: N-gram x Classifier"); plt.tight_layout(); plt.show()

## Top N-grams by Class

In [ ]:
tfidf_bi=TfidfVectorizer(max_features=10000,ngram_range=(2,2))
X_all=tfidf_bi.fit_transform(df["text"]); terms=tfidf_bi.get_feature_names_out()
for label,name in enumerate(CLASS_NAMES):
    indices=np.where(df["label"].values==label)[0]; avg=np.asarray(X_all[indices].mean(axis=0)).flatten()
    top_idx=avg.argsort()[-10:][::-1]
    print(f"\nTop 10 bigrams for {name}:")
    for t,s in zip([terms[i] for i in top_idx],avg[top_idx]): print(f"  {t:30s} ({s:.4f})")

## LazyPredict Benchmark

> Not used for NLP tasks per best practices.

## FLAML AutoML

> Not used for NLP tasks per best practices.

## Additional - CountVectorizer vs TfidfVectorizer

In [ ]:
vecs={"CountVectorizer":CountVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2)),"TfidfVectorizer":TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2))}
vr=[]
for vn,v in vecs.items():
    X_tr=v.fit_transform(X_train); X_te=v.transform(X_test)
    for cn,clf in classifiers.items():
        m=clone(clf); m.fit(X_tr,y_train); yp=m.predict(X_te)
        vr.append({"Vec":vn,"Clf":cn,"Acc":accuracy_score(y_test,yp),"F1":f1_score(y_test,yp)})
print(pd.DataFrame(vr).to_string(index=False))

## Top 3 Model Selection

In [ ]:
top3=results_df.sort_values("F1",ascending=False).head(3).reset_index(drop=True)
print(top3[["N-gram","Classifier","Accuracy","F1"]].to_string(index=False))
print(f"Baseline F1={bl_f1:.4f}, Improvement: {((top3.iloc[0]['F1']-bl_f1)/bl_f1)*100:+.2f}%")

## Final Training and Evaluation of Top 3

In [ ]:
top3_preds={}
for i,row in top3.iterrows():
    ng=NGRAM_CONFIGS[row["N-gram"]]
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=ng)
    X_tr=tfidf.fit_transform(X_train); X_te=tfidf.transform(X_test)
    if row["Classifier"]=="MultinomialNB": m=MultinomialNB()
    elif row["Classifier"]=="LogisticRegression": m=LogisticRegression(max_iter=1000,random_state=SEED)
    else: m=LinearSVC(max_iter=2000,random_state=SEED)
    m.fit(X_tr,y_train); yp=m.predict(X_te)
    top3_preds[f"{row['N-gram']}+{row['Classifier']}"]=yp
    print(f"\n{'='*60}\n#{i+1}\n{'='*60}")
    print(classification_report(y_test,yp,target_names=CLASS_NAMES))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
for idx,(n,yp) in enumerate(top3_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,yp),display_labels=CLASS_NAMES).plot(ax=axes[idx],cmap="Blues",values_format="d")
    axes[idx].set_title(f"#{idx+1}",fontsize=9)
plt.tight_layout(); plt.show()

## Error Analysis

In [ ]:
bp=list(top3_preds.values())[0]; em=bp!=y_test.values
edf=pd.DataFrame({"text":X_test.values[em],"true":y_test.values[em],"pred":bp[em]})
print(f"Errors: {len(edf)}/{len(y_test)} ({len(edf)/len(y_test)*100:.1f}%)")

## Interpretation and Business Insight

1. Bigrams (1,2) provide the best balance.
2. Trigrams offer diminishing returns.
3. Pure bi/trigrams miss individual word signals.
4. Bigrams capture negation (not good).
5. max_features regularizes effectively.

## Limitations

Single dataset. Fixed max_features. TF-IDF only. English only. No char n-grams.

## How to Improve This Project

1. Char n-grams. 2. Mutual information selection. 3. Topic classification. 4. Cross-validation.

## Production Considerations

(1,2) is a good default. max_features controls memory. HashingVectorizer for scale. Char n-grams for noisy text.

## Common Mistakes

1. Using only trigrams without unigrams.
2. No feature cap (memory issues).
3. Ignoring sparsity.
4. Not lowercasing.
5. Forgetting transformers handle context natively.

## Mini Challenge / Exercises

1. Try character-level n-grams.
2. Find optimal max_features.
3. Top trigrams per class.
4. Short vs long text comparison.

## Final Summary / Key Takeaways

1. Start with (1,2).
2. Trigrams rarely justify the cost.
3. Always include unigrams.
4. max_features prevents overfitting.
5. Bigrams capture negation and phrases.